In [1]:
import pandas as pd
import numpy as np
import re
import string
import nltk

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

In [2]:
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\tanis\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\tanis\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [3]:
train_df = pd.read_csv("../data/raw/sent_train.csv")
valid_df = pd.read_csv("../data/raw/sent_valid.csv")

print(train_df.shape)
print(valid_df.shape)

(9543, 2)
(2388, 2)


In [4]:
print(train_df.columns)

Index(['text', 'label'], dtype='object')


In [5]:
train_df["label"].value_counts()

label
2    6178
1    1923
0    1442
Name: count, dtype: int64

In [6]:
train_df[["text", "label"]].head(10)

,text,label
0,$BYND - JPMorgan reels in expectations on Beyo...,0
1,$CCL $RCL - Nomura points to bookings weakness...,0
2,"$CX - Cemex cut at Credit Suisse, J.P. Morgan ...",0
3,$ESS: BTIG Research cuts to Neutral https://t....,0
4,$FNKO - Funko slides after Piper Jaffray PT cu...,0
5,$FTI - TechnipFMC downgraded at Berenberg but ...,0
6,$GM - GM loses a bull https://t.co/tdUfG5HbXy,0
7,$GM: Deutsche Bank cuts to Hold https://t.co/7...,0
8,$GTT: Cowen cuts to Market Perform,0
9,$HNHAF $HNHPD $AAPL - Trendforce cuts iPhone e...,0


In [7]:
stop_words = set(stopwords.words("english"))

print("Total Stopwords:", len(stop_words))

Total Stopwords: 198


In [8]:
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\tanis\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [9]:
def clean_text(text):

    # convert to lowercase
    text = text.lower()

    # remove urls
    text = re.sub(r"http\S+|www\S+|https\S+", "", text)

    # keep stock tickers but remove $
    text = re.sub(r"\$", "", text)

    # remove punctuation
    text = text.translate(str.maketrans("", "", string.punctuation))

    # remove numbers
    text = re.sub(r"\d+", "", text)

    # tokenize
    tokens = word_tokenize(text)

    # remove stopwords
    tokens = [word for word in tokens if word not in stop_words]

    # join back
    cleaned_text = " ".join(tokens)

    return cleaned_text

In [10]:
train_df["clean_text"] = train_df["text"].apply(clean_text)
valid_df["clean_text"] = valid_df["text"].apply(clean_text)

In [11]:
for i in range(5):

    print("=" * 80)

    print("ORIGINAL TEXT:\n")
    print(train_df["text"][i])

    print("\nCLEANED TEXT:\n")
    print(train_df["clean_text"][i])

    print("\n")

ORIGINAL TEXT:

$BYND - JPMorgan reels in expectations on Beyond Meat https://t.co/bd0xbFGjkT

CLEANED TEXT:

bynd jpmorgan reels expectations beyond meat


ORIGINAL TEXT:

$CCL $RCL - Nomura points to bookings weakness at Carnival and Royal Caribbean https://t.co/yGjpT2ReD3

CLEANED TEXT:

ccl rcl nomura points bookings weakness carnival royal caribbean


ORIGINAL TEXT:

$CX - Cemex cut at Credit Suisse, J.P. Morgan on weak building outlook https://t.co/KN1g4AWFIb

CLEANED TEXT:

cx cemex cut credit suisse jp morgan weak building outlook


ORIGINAL TEXT:

$ESS: BTIG Research cuts to Neutral https://t.co/MCyfTsXc2N

CLEANED TEXT:

ess btig research cuts neutral


ORIGINAL TEXT:

$FNKO - Funko slides after Piper Jaffray PT cut https://t.co/z37IJmCQzB

CLEANED TEXT:

fnko funko slides piper jaffray pt cut




In [12]:
empty_rows = train_df[train_df["clean_text"].str.strip() == ""]

print("Empty Rows:", len(empty_rows))

Empty Rows: 4


In [13]:
# remove empty rows from training data
train_df = train_df[train_df["clean_text"].str.strip() != ""]

# remove empty rows from validation data
valid_df = valid_df[valid_df["clean_text"].str.strip() != ""]

# reset indexes
train_df.reset_index(drop=True, inplace=True)
valid_df.reset_index(drop=True, inplace=True)

print("Train Shape After Cleaning:", train_df.shape)
print("Validation Shape After Cleaning:", valid_df.shape)

Train Shape After Cleaning: (9539, 3)
Validation Shape After Cleaning: (2388, 3)


In [14]:
train_df.head()

,text,label,clean_text
0,$BYND - JPMorgan reels in expectations on Beyo...,0,bynd jpmorgan reels expectations beyond meat
1,$CCL $RCL - Nomura points to bookings weakness...,0,ccl rcl nomura points bookings weakness carniv...
2,"$CX - Cemex cut at Credit Suisse, J.P. Morgan ...",0,cx cemex cut credit suisse jp morgan weak buil...
3,$ESS: BTIG Research cuts to Neutral https://t....,0,ess btig research cuts neutral
4,$FNKO - Funko slides after Piper Jaffray PT cu...,0,fnko funko slides piper jaffray pt cut


In [15]:
train_df.to_csv("../data/processed/train_processed.csv", index=False)
valid_df.to_csv("../data/processed/valid_processed.csv", index=False)

print("Processed datasets saved successfully.")

Processed datasets saved successfully.
